# 00 — Source Configuration and Reconciliation

**Pipeline:** Skin Cancer 3-Class Classification (NV / MEL / BCC)  
**Purpose:** Define source paths, create output folders, scan raw class folders, load the authoritative ground truth CSV, reconcile file identities and labels, and save a clean reconciled manifest.  

**Out of scope for this notebook:** metadata merge, duplicate exclusion, quality filtering, preprocessing, augmentation, splitting, training.

---

## Section 0 — Imports

In [2]:
import json
import os
from pathlib import Path

import pandas as pd

## Section 1 — Frozen Path Configuration

In [3]:
# Frozen source paths
RAW_DATASET_ROOT   = Path(r"C:\SKIN CANCER v2\DS")
NV_DIR             = Path(r"C:\SKIN CANCER v2\DS\NV")
BCC_DIR            = Path(r"C:\SKIN CANCER v2\DS\BCC")
MEL_DIR            = Path(r"C:\SKIN CANCER v2\DS\MEL")
GROUND_TRUTH_CSV   = Path(r"C:\SKIN CANCER v2\DS\ISIC_2019_Training_GroundTruth.csv")
METADATA_CSV       = Path(r"C:\SKIN CANCER v2\DS\ISIC_2019_Training_Metadata.csv")
FINAL_DATASET_ROOT = Path(r"C:\SKIN CANCER v2\final DS")
OUTPUT_ROOT        = Path(r"C:\SKIN CANCER v2\pipe output")

# Class mapping
CLASS_LABELS = ["NV", "MEL", "BCC"]
CLASS_INDEX  = {"NV": 0, "MEL": 1, "BCC": 2}

# Per-class source directories
CLASS_DIRS = {
    "NV":  NV_DIR,
    "MEL": MEL_DIR,
    "BCC": BCC_DIR,
}

# Allowed image extensions (lower-case)
ALLOWED_EXTENSIONS = {".jpg", ".jpeg", ".png"}

# Approved filename suffixes that may be stripped to recover a ground-truth ID
APPROVED_SUFFIXES = ["_downsampled"]

print("Path configuration loaded.")
for label, d in CLASS_DIRS.items():
    print(f"  {label}: {d}")
print(f"  Ground truth CSV : {GROUND_TRUTH_CSV}")
print(f"  Output root      : {OUTPUT_ROOT}")

Path configuration loaded.
  NV: C:\SKIN CANCER v2\DS\NV
  MEL: C:\SKIN CANCER v2\DS\MEL
  BCC: C:\SKIN CANCER v2\DS\BCC
  Ground truth CSV : C:\SKIN CANCER v2\DS\ISIC_2019_Training_GroundTruth.csv
  Output root      : C:\SKIN CANCER v2\pipe output


## Section 2 — Create Output Folders

In [4]:
OUTPUT_CONFIG_DIR         = OUTPUT_ROOT / "00_config"
OUTPUT_RECONCILIATION_DIR = OUTPUT_ROOT / "01_reconciliation"

for folder in [OUTPUT_CONFIG_DIR, OUTPUT_RECONCILIATION_DIR]:
    folder.mkdir(parents=True, exist_ok=True)
    print(f"Ready: {folder}")

Ready: C:\SKIN CANCER v2\pipe output\00_config
Ready: C:\SKIN CANCER v2\pipe output\01_reconciliation


## Section 3 — Save Source Configuration JSON

In [5]:
source_config = {
    "raw_dataset_root":   str(RAW_DATASET_ROOT),
    "class_dirs": {
        label: str(path) for label, path in CLASS_DIRS.items()
    },
    "ground_truth_csv":   str(GROUND_TRUTH_CSV),
    "metadata_csv":       str(METADATA_CSV),
    "final_dataset_root": str(FINAL_DATASET_ROOT),
    "output_root":        str(OUTPUT_ROOT),
    "class_mapping":      CLASS_INDEX,
    "allowed_extensions": sorted(ALLOWED_EXTENSIONS),
    "approved_suffixes":  APPROVED_SUFFIXES,
}

config_path = OUTPUT_CONFIG_DIR / "source_config.json"
with open(config_path, "w", encoding="utf-8") as f:
    json.dump(source_config, f, indent=2)

print(f"source_config.json saved -> {config_path}")

source_config.json saved -> C:\SKIN CANCER v2\pipe output\00_config\source_config.json


## Section 4 — Scan Raw Class Folders

In [6]:
def scan_class_folder(label, folder):
    records = []
    if not folder.exists():
        print(f"  WARNING: folder does not exist -- {folder}")
        return records

    for entry in sorted(folder.iterdir()):
        if not entry.is_file():
            continue
        ext = entry.suffix.lower()
        if ext not in ALLOWED_EXTENSIONS:
            continue

        stem_raw = entry.stem

        recognized_suffix = ""
        base_id_candidate = stem_raw
        for suffix in APPROVED_SUFFIXES:
            if stem_raw.endswith(suffix):
                recognized_suffix = suffix
                base_id_candidate = stem_raw[: -len(suffix)]
                break

        records.append({
            "full_path":                str(entry),
            "folder_name":              label,
            "file_name_with_extension": entry.name,
            "file_stem_raw":            stem_raw,
            "extension":                ext,
            "recognized_suffix":        recognized_suffix,
            "base_id_candidate":        base_id_candidate,
            "file_exists":              True,
            "file_size_bytes":          entry.stat().st_size,
            "folder_label":             label,
        })
    return records


all_records = []
scan_counts = {}

for label, folder in CLASS_DIRS.items():
    recs = scan_class_folder(label, folder)
    scan_counts[label] = len(recs)
    all_records.extend(recs)
    print(f"  {label}: {len(recs):,} images scanned from {folder}")

print(f"\nTotal raw images scanned: {len(all_records):,}")

  NV: 12,875 images scanned from C:\SKIN CANCER v2\DS\NV
  MEL: 4,522 images scanned from C:\SKIN CANCER v2\DS\MEL
  BCC: 3,323 images scanned from C:\SKIN CANCER v2\DS\BCC

Total raw images scanned: 20,720


## Section 5 — Load and Validate Ground Truth CSV

In [7]:
gt_raw = pd.read_csv(GROUND_TRUTH_CSV)
print(f"Ground truth CSV loaded: {len(gt_raw):,} rows, columns: {list(gt_raw.columns)}")

Ground truth CSV loaded: 25,331 rows, columns: ['image', 'MEL', 'NV', 'BCC', 'AK', 'BKL', 'DF', 'VASC', 'SCC', 'UNK']


In [8]:
# Detect image-ID column (first column)
id_col = gt_raw.columns[0]
print(f"Image ID column detected: '{id_col}'")

available_class_cols = [c for c in CLASS_LABELS if c in gt_raw.columns]
missing_class_cols   = [c for c in CLASS_LABELS if c not in gt_raw.columns]

if missing_class_cols:
    print(f"  WARNING: expected class columns not found in CSV: {missing_class_cols}")
print(f"  Class columns found: {available_class_cols}")

gt = gt_raw[[id_col] + available_class_cols].copy()
gt = gt.rename(columns={id_col: "image_id"})

for col in available_class_cols:
    gt[col] = pd.to_numeric(gt[col], errors="coerce").fillna(0).astype(int)

gt["active_label_count"] = gt[available_class_cols].sum(axis=1)

def derive_gt_label(row):
    if row["active_label_count"] == 1:
        for cls in available_class_cols:
            if row[cls] == 1:
                return cls
    return None

gt["gt_label"] = gt.apply(derive_gt_label, axis=1)

invalid_gt = gt[gt["gt_label"].isna()]
valid_gt   = gt[gt["gt_label"].notna()]

print(f"\nGround truth rows total   : {len(gt):,}")
print(f"  Valid (exactly 1 label) : {len(valid_gt):,}")
print(f"  Invalid (0 or >1 label) : {len(invalid_gt):,}")

gt_lookup     = dict(zip(valid_gt["image_id"], valid_gt["gt_label"]))
invalid_gt_ids = set(invalid_gt["image_id"])

print(f"  GT lookup entries       : {len(gt_lookup):,}")

Image ID column detected: 'image'
  Class columns found: ['NV', 'MEL', 'BCC']

Ground truth rows total   : 25,331
  Valid (exactly 1 label) : 20,720
  Invalid (0 or >1 label) : 4,611
  GT lookup entries       : 20,720


## Section 6 — Reconcile Files Against Ground Truth

In [9]:
def reconcile_record(rec):
    stem_raw     = rec["file_stem_raw"]
    base_id      = rec["base_id_candidate"]
    folder_label = rec["folder_label"]

    # Match against GT lookup
    exact_hit  = stem_raw in gt_lookup or stem_raw in invalid_gt_ids
    suffix_hit = (stem_raw != base_id) and (
        base_id in gt_lookup or base_id in invalid_gt_ids
    )

    if exact_hit and suffix_hit:
        match_status = "ambiguous_match"
        canonical_id = stem_raw
    elif exact_hit:
        match_status = "matched_exact"
        canonical_id = stem_raw
    elif suffix_hit:
        match_status = "matched_via_suffix_rule"
        canonical_id = base_id
    else:
        match_status = "unmatched_ground_truth"
        canonical_id = ""

    # Retrieve GT label
    if match_status == "unmatched_ground_truth":
        gt_label               = None
        label_agreement_status = "missing_ground_truth"
    elif canonical_id in invalid_gt_ids:
        gt_label               = None
        label_agreement_status = "invalid_ground_truth"
    else:
        gt_label = gt_lookup.get(canonical_id)
        if gt_label is None:
            label_agreement_status = "invalid_ground_truth"
        elif gt_label == folder_label:
            label_agreement_status = "agree"
        else:
            label_agreement_status = "disagree"

    # Authoritative label and class index
    final_label = gt_label
    class_idx   = CLASS_INDEX.get(final_label) if final_label else None

    # Eligibility
    issue_parts = []
    if match_status in ("unmatched_ground_truth", "ambiguous_match"):
        issue_parts.append(match_status)
    if label_agreement_status in ("missing_ground_truth", "invalid_ground_truth", "disagree"):
        issue_parts.append(label_agreement_status)
    if final_label not in CLASS_LABELS:
        issue_parts.append("label_out_of_scope")

    eligible     = len(issue_parts) == 0
    issue_reason = "; ".join(issue_parts) if issue_parts else ""

    return {
        **rec,
        "canonical_match_id":             canonical_id,
        "gt_label":                        gt_label,
        "final_authoritative_label":       final_label,
        "class_index":                     class_idx,
        "match_status":                    match_status,
        "label_agreement_status":          label_agreement_status,
        "eligible_after_reconciliation":   eligible,
        "reconciliation_issue_reason":     issue_reason,
    }


reconciled_records = [reconcile_record(r) for r in all_records]
print(f"Reconciliation complete: {len(reconciled_records):,} records processed.")

Reconciliation complete: 20,720 records processed.


## Section 7 — Build Manifest DataFrame

In [10]:
MANIFEST_COLUMNS = [
    "full_path",
    "folder_name",
    "file_name_with_extension",
    "file_stem_raw",
    "extension",
    "recognized_suffix",
    "base_id_candidate",
    "canonical_match_id",
    "folder_label",
    "gt_label",
    "final_authoritative_label",
    "class_index",
    "match_status",
    "label_agreement_status",
    "file_exists",
    "file_size_bytes",
    "eligible_after_reconciliation",
    "reconciliation_issue_reason",
]

manifest_df = pd.DataFrame(reconciled_records)

for col in MANIFEST_COLUMNS:
    if col not in manifest_df.columns:
        manifest_df[col] = None

manifest_df = manifest_df[MANIFEST_COLUMNS]

print(f"Manifest shape: {manifest_df.shape}")
manifest_df.head(3)

Manifest shape: (20720, 18)


,full_path,folder_name,file_name_with_extension,file_stem_raw,extension,recognized_suffix,base_id_candidate,canonical_match_id,folder_label,gt_label,final_authoritative_label,class_index,match_status,label_agreement_status,file_exists,file_size_bytes,eligible_after_reconciliation,reconciliation_issue_reason
0,C:\SKIN CANCER v2\DS\NV\ISIC_0000000.jpg,NV,ISIC_0000000.jpg,ISIC_0000000,.jpg,,ISIC_0000000,ISIC_0000000,NV,NV,NV,0,matched_exact,agree,True,49964,True,
1,C:\SKIN CANCER v2\DS\NV\ISIC_0000001.jpg,NV,ISIC_0000001.jpg,ISIC_0000001,.jpg,,ISIC_0000001,ISIC_0000001,NV,NV,NV,0,matched_exact,agree,True,38941,True,
2,C:\SKIN CANCER v2\DS\NV\ISIC_0000003.jpg,NV,ISIC_0000003.jpg,ISIC_0000003,.jpg,,ISIC_0000003,ISIC_0000003,NV,NV,NV,0,matched_exact,agree,True,45774,True,


## Section 8 — Validation Summary

In [11]:
total_scanned    = len(manifest_df)
gt_total_rows    = len(gt)
gt_invalid_rows  = len(invalid_gt)
matched_exact_n  = (manifest_df["match_status"] == "matched_exact").sum()
matched_suffix_n = (manifest_df["match_status"] == "matched_via_suffix_rule").sum()
unmatched_n      = (manifest_df["match_status"] == "unmatched_ground_truth").sum()
ambiguous_n      = (manifest_df["match_status"] == "ambiguous_match").sum()
disagreements_n  = (manifest_df["label_agreement_status"] == "disagree").sum()
eligible_df      = manifest_df[manifest_df["eligible_after_reconciliation"] == True]
eligible_total   = len(eligible_df)
eligible_by_class = eligible_df["final_authoritative_label"].value_counts()

print("=" * 55)
print("  RECONCILIATION VALIDATION SUMMARY")
print("=" * 55)
print(f"  Raw scanned images (total)     : {total_scanned:>8,}")
for label in CLASS_LABELS:
    print(f"    {label:<5}                        : {scan_counts.get(label, 0):>8,}")
print("-" * 55)
print(f"  Ground truth rows (total)      : {gt_total_rows:>8,}")
print(f"  Invalid GT rows (0 or >1 label): {gt_invalid_rows:>8,}")
print("-" * 55)
print(f"  matched_exact                  : {matched_exact_n:>8,}")
print(f"  matched_via_suffix_rule        : {matched_suffix_n:>8,}")
print(f"  unmatched_ground_truth         : {unmatched_n:>8,}")
print(f"  ambiguous_match                : {ambiguous_n:>8,}")
print("-" * 55)
print(f"  Folder/GT label disagreements  : {disagreements_n:>8,}")
print("-" * 55)
print(f"  ELIGIBLE after reconciliation  : {eligible_total:>8,}")
for label in CLASS_LABELS:
    count = eligible_by_class.get(label, 0)
    print(f"    {label:<5}                        : {count:>8,}")
print("=" * 55)

  RECONCILIATION VALIDATION SUMMARY
  Raw scanned images (total)     :   20,720
    NV                           :   12,875
    MEL                          :    4,522
    BCC                          :    3,323
-------------------------------------------------------
  Ground truth rows (total)      :   25,331
  Invalid GT rows (0 or >1 label):    4,611
-------------------------------------------------------
  matched_exact                  :   20,720
  matched_via_suffix_rule        :        0
  unmatched_ground_truth         :        0
  ambiguous_match                :        0
-------------------------------------------------------
  Folder/GT label disagreements  :        0
-------------------------------------------------------
  ELIGIBLE after reconciliation  :   20,720
    NV                           :   12,875
    MEL                          :    4,522
    BCC                          :    3,323


## Section 9 — Save Outputs

In [12]:
# 1. Reconciled manifest (all rows, including ineligible)
manifest_path = OUTPUT_RECONCILIATION_DIR / "01_reconciled_manifest.csv"
manifest_df.to_csv(manifest_path, index=False)
print(f"Saved manifest ({len(manifest_df):,} rows) -> {manifest_path}")

Saved manifest (20,720 rows) -> C:\SKIN CANCER v2\pipe output\01_reconciliation\01_reconciled_manifest.csv


In [13]:
# 2. Reconciliation summary
summary_rows = []
for label in CLASS_LABELS:
    summary_rows.append({"metric": f"scanned_{label}", "value": scan_counts.get(label, 0)})

summary_rows.extend([
    {"metric": "scanned_total",                "value": total_scanned},
    {"metric": "ground_truth_rows_total",       "value": gt_total_rows},
    {"metric": "ground_truth_rows_invalid",     "value": gt_invalid_rows},
    {"metric": "matched_exact",                 "value": int(matched_exact_n)},
    {"metric": "matched_via_suffix_rule",       "value": int(matched_suffix_n)},
    {"metric": "unmatched_ground_truth",        "value": int(unmatched_n)},
    {"metric": "ambiguous_match",               "value": int(ambiguous_n)},
    {"metric": "label_disagreements",           "value": int(disagreements_n)},
    {"metric": "eligible_total",                "value": eligible_total},
])
for label in CLASS_LABELS:
    summary_rows.append({"metric": f"eligible_{label}", "value": int(eligible_by_class.get(label, 0))})

summary_df = pd.DataFrame(summary_rows)
summary_path = OUTPUT_RECONCILIATION_DIR / "01_reconciliation_summary.csv"
summary_df.to_csv(summary_path, index=False)
print(f"Saved summary ({len(summary_df)} rows) -> {summary_path}")

Saved summary (15 rows) -> C:\SKIN CANCER v2\pipe output\01_reconciliation\01_reconciliation_summary.csv


In [14]:
# 3. Reconciliation issues (ineligible rows only)
issues_df = manifest_df[manifest_df["eligible_after_reconciliation"] == False].copy()
issues_path = OUTPUT_RECONCILIATION_DIR / "01_reconciliation_issues.csv"
issues_df.to_csv(issues_path, index=False)
print(f"Saved issues ({len(issues_df):,} rows) -> {issues_path}")

Saved issues (0 rows) -> C:\SKIN CANCER v2\pipe output\01_reconciliation\01_reconciliation_issues.csv


## Section 10 — Output File Verification

In [15]:
output_files = [
    OUTPUT_CONFIG_DIR / "source_config.json",
    OUTPUT_RECONCILIATION_DIR / "01_reconciled_manifest.csv",
    OUTPUT_RECONCILIATION_DIR / "01_reconciliation_summary.csv",
    OUTPUT_RECONCILIATION_DIR / "01_reconciliation_issues.csv",
]

print("Output file verification:")
all_ok = True
for p in output_files:
    exists = p.exists()
    size   = p.stat().st_size if exists else 0
    status = "OK" if exists else "MISSING"
    print(f"  [{status}] {p.name:<45}  {size:>10,} bytes")
    if not exists:
        all_ok = False

if all_ok:
    print("\nAll output files present.")
else:
    print("\nWARNING: one or more output files are missing.")

Output file verification:
  [OK] source_config.json                                    671 bytes
  [OK] 01_reconciled_manifest.csv                      3,373,046 bytes
  [OK] 01_reconciliation_summary.csv                         347 bytes
  [OK] 01_reconciliation_issues.csv                          309 bytes

All output files present.


## Section 11 — Completion Summary

**Section 00 — Source Configuration and Reconciliation is complete.**

What was accomplished in this notebook:
- All source paths frozen and saved to `source_config.json`.
- Output folders `00_config/` and `01_reconciliation/` created under the pipeline output root.
- Raw class folders (NV, MEL, BCC) scanned; file identities recorded without touching source files.
- Authoritative ground truth CSV loaded and validated; invalid rows (zero or multiple active labels) identified.
- Each scanned file reconciled against ground truth: match status, GT label, and label agreement recorded.
- All rows saved to the reconciled manifest (eligible and ineligible rows both retained).
- Reconciliation summary and issues files saved.

**What was deliberately deferred:**
- Metadata merge (ISIC_2019_Training_Metadata.csv)
- Duplicate family detection and exclusion
- Quality audit (image integrity, resolution checks)
- Preprocessing, augmentation, and train/val/test splitting

**Next notebook:** `01_metadata_duplicates_and_quality.ipynb`  
That notebook should merge ISIC metadata onto the reconciled manifest, identify duplicate image families, and perform a quality audit (file integrity, resolution, corruption checks) to produce a final clean dataset ready for splitting and training.